# Anime Shorts Generator (DUAL GPU Worker)
This script uses BOTH T4 GPUs simultaneously to render Anime videos twice as fast, adds Edge-TTS voiceovers, burns Whisper subtitles, and sends Email notifications.

In [ ]:
# 1. Install Heavy Requirements
!pip install diffusers transformers accelerate safetensors edge-tts openai-whisper moviepy imageio[ffmpeg] google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
# 2. Fetch the AI Storyboard
import urllib.request
import json
import os

url = 'https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/outputs/latest_script.json'
try:
    response = urllib.request.urlopen(url)
    script_data = json.loads(response.read().decode('utf-8'))
    print(f"Processing Anime Short: {script_data['title']}")
except:
    # Fallback mock data for testing if url isn't set up yet
    script_data = {"title": "Test", "scenes": [{"visual_prompt": "1boy, anime", "narration": "Testing"}, {"visual_prompt": "1girl, anime", "narration": "GPU 2"}]}
    
os.makedirs("/kaggle/working/assets", exist_ok=True)

In [ ]:
# 3. Initialize AnimateDiff on BOTH GPUs (Parallel Processing)
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_video

print("Loading MeinaMix Anime Model on GPU 0...")
adapter0 = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
pipe0 = AnimateDiffPipeline.from_pretrained("Meina/MeinaMix_V11", motion_adapter=adapter0, torch_dtype=torch.float16).to("cuda:0")
pipe0.scheduler = EulerDiscreteScheduler.from_config(pipe0.scheduler.config)

print("Loading MeinaMix Anime Model on GPU 1...")
adapter1 = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
pipe1 = AnimateDiffPipeline.from_pretrained("Meina/MeinaMix_V11", motion_adapter=adapter1, torch_dtype=torch.float16).to("cuda:1")
pipe1.scheduler = EulerDiscreteScheduler.from_config(pipe1.scheduler.config)


In [ ]:
# 4. Generate Scenes in Parallel
import subprocess
import concurrent.futures

scenes = script_data.get('scenes', [])

def generate_scene(task_data):
    i, scene, gpu_id = task_data
    print(f"[GPU {gpu_id}] Starting Scene {i+1}...")
    
    # Select the correct GPU pipeline
    pipe = pipe0 if gpu_id == 0 else pipe1
    
    # A. Audio (Edge-TTS)
    audio_path = f"/kaggle/working/assets/audio_{i}.mp3"
    text = scene['narration'].replace("'", "")
    subprocess.run(["edge-tts", "--voice", "en-US-ChristopherNeural", "--text", text, "--write-media", audio_path])
    
    # B. Video (AnimateDiff)
    video_path = f"/kaggle/working/assets/video_{i}.mp4"
    prompt = scene['visual_prompt']
    # Generate frames
    output = pipe(prompt=prompt, negative_prompt="bad quality, worst quality, deformed", num_frames=16, guidance_scale=7.5, num_inference_steps=25).frames[0]
    export_to_video(output, video_path, fps=8)
    
    print(f"[GPU {gpu_id}] Finished Scene {i+1}!")
    return (i, video_path, audio_path)

# Distribute tasks: Evens on GPU 0, Odds on GPU 1
tasks = [(i, scene, i % 2) for i, scene in enumerate(scenes)]
video_clips = []

print("Starting parallel video generation...")
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(generate_scene, tasks))

# Sort results back into correct order
results.sort(key=lambda x: x[0])
video_clips = [(vid, aud) for idx, vid, aud in results]
print("All scenes generated!")

In [ ]:
# 5. Free VRAM, Load Whisper, & Edit Video
import gc
import whisper
from moviepy.editor import VideoFileClip, AudioFileClip, TextClip, CompositeVideoClip, concatenate_videoclips

# Clean up GPU memory so Whisper has room
del pipe0
del pipe1
torch.cuda.empty_cache()
gc.collect()

print("Loading highly-accurate Whisper Medium model on GPU 0...")
whisper_model = whisper.load_model("medium").to("cuda:0")

final_clips = []
for vid_path, aud_path in video_clips:
    v_clip = VideoFileClip(vid_path)
    a_clip = AudioFileClip(aud_path)
    
    # Loop video if audio is longer
    if a_clip.duration > v_clip.duration:
        from moviepy.video.fx.all import loop
        v_clip = loop(v_clip, duration=a_clip.duration)
    else:
        v_clip = v_clip.subclip(0, a_clip.duration)
        
    v_clip = v_clip.set_audio(a_clip)
    
    # Whisper Subtitles
    result = whisper_model.transcribe(aud_path, word_timestamps=True)
    subs = []
    for segment in result['segments']:
        for word in segment['words']:
            txt_clip = TextClip(word['word'].strip(), fontsize=50, color='yellow', font='Impact', stroke_color='black', stroke_width=2)
            txt_clip = txt_clip.set_position(('center', 'center')).set_start(word['start']).set_end(word['end'])
            subs.append(txt_clip)
            
    scene_final = CompositeVideoClip([v_clip] + subs)
    final_clips.append(scene_final)

final_video = concatenate_videoclips(final_clips)
output_path = "/kaggle/working/final_anime_short.mp4"
final_video.write_videofile(output_path, fps=24)
print("Video Editing Complete!")

In [ ]:
# 6. Upload to Drive & Email Notification
import smtplib
from email.message import EmailMessage
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

try:
    email_user = user_secrets.get_secret("EMAIL_USER")
    email_pass = user_secrets.get_secret("EMAIL_PASS")
    
    msg = EmailMessage()
    msg['Subject'] = script_data.get('email_subject', 'New Anime Video Ready!')
    msg['From'] = email_user
    msg['To'] = email_user
    
    body = f"{script_data.get('email_body', 'Your video is ready.')}\n\nTitle: {script_data['title']}\nDescription: {script_data['description']}\nTags: {', '.join(script_data['tags'])}"
    msg.set_content(body)
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(email_user, email_pass)
        smtp.send_message(msg)
    print("Email notification sent successfully!")
except Exception as e:
    print("Notification Skipped. Error:", e)